# MTH9877 — Assignment 3: Part E(ii) — Time-Varying Covariates

**Standalone notebook.**  Requires the three processed parquet files in `processed/`.

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from lifelines import (
    AalenJohansenFitter,
    KaplanMeierFitter,
    CoxPHFitter,
    CoxTimeVaryingFitter,
)
from lifelines.utils import concordance_index
from scipy.ndimage import gaussian_filter1d
warnings.filterwarnings("ignore")

BASE          = Path("/Users/yueqilin/Desktop/MTH9877 IR/IR&C Assignment3")
OUT_DIR       = BASE / "processed"
SURVIVAL_PATH = OUT_DIR / "survival_loans.parquet"
MACRO_PATH    = OUT_DIR / "macro_monthly.parquet"
PANEL_PATH    = OUT_DIR / "panel_monthly.parquet"

DEVICE = (
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("cpu")
)
print(f"Device : {DEVICE}")
for p in [SURVIVAL_PATH, MACRO_PATH, PANEL_PATH]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {p.name}")

Device : mps
  [OK] survival_loans.parquet
  [OK] macro_monthly.parquet
  [OK] panel_monthly.parquet


## Setup — Load Data & Train Models

Loads the three parquet files and builds the working datasets used throughout:

| Variable | Contents | Used by |
|---|---|---|
| `survival` | 34M loans, one row each | EDA (full dataset) |
| `sv_sub` | 100K stratified sample | AJ CIF, Cox models, scenario analysis |
| `macro` | Monthly macro (1999–2025) | Deep Cox, cause-specific Cox, E(ii) |
| `dc_df` | sv_sub + annual macro | Deep Cox training |
| `FEATURES` / `xgb_model` | Panel-trained XGBoost | E(iv) scenario analysis |


In [2]:
# ── Survival dataset + 100K stratified subsample ─────────────────────────────
survival = pl.read_parquet(SURVIVAL_PATH)
print(f"Full dataset : {survival.height:,} loans")

B_SAMPLE_N = 100_000
sv_sub = (
    survival
    .with_columns(pl.col("VintageYear").cast(pl.Int32))
    .group_by("VintageYear")
    .map_groups(lambda g: g.sample(
        n=min(len(g), max(1, int(B_SAMPLE_N * len(g) / survival.height))),
        seed=42
    ))
).to_pandas()
print(f"Subsample    : {len(sv_sub):,} loans  (stratified by vintage year)")

# ── Macro covariates ──────────────────────────────────────────────────────────
macro = pl.read_parquet(MACRO_PATH)
print(f"Macro rows   : {macro.height:,}  columns: {macro.columns}")

---
## E(ii) — Time-Dependent Covariates

Standard Cox models fix covariates at origination. In reality the key drivers of
prepayment and default — **mortgage rates, unemployment, and the rate incentive** —
evolve every month. The **Andersen-Gill counting-process** extension handles this:
each loan contributes one row per calendar period, with that period's covariate values.

| Model | Covariate assumption | Data format |
|---|---|---|
| Standard Cox — E(i)(e) | Fixed at origination vintage | One row per loan |
| **Andersen-Gill Cox — E(ii)** | Updated each month | One row per loan-month |

**Key question**: does using *current* `rate_incentive` (orig_rate − today's rate) rather
than the origination-vintage snapshot improve the prepayment hazard estimate?

**Data**: 10,000 loans sampled from the monthly panel (stratified by vintage year),
yielding ~500K loan-month rows with `(tstart, tstop, event)` intervals per loan.

In [14]:
TV_LOANS = 50_000

# ── Sample 50K loans from panel (stratified by vintage_year) ──────────────────
panel_full = pl.read_parquet(PANEL_PATH)
loan_meta = (
    panel_full
    .group_by("loan_id")
    .agg(pl.col("vintage_year").first())
)
tv_loan_ids = (
    loan_meta
    .group_by("vintage_year")
    .map_groups(lambda g: g.sample(
        n=min(len(g), max(1, int(TV_LOANS * len(g) / loan_meta.height))),
        seed=42
    ))
)["loan_id"].to_list()
print(f"TV loans sampled: {len(tv_loan_ids):,}")

# ── Filter panel and attach default event from survival ────────────────────────
surv_events = (
    pl.read_parquet(SURVIVAL_PATH)
    .select(["LoanSequenceNumber", "defaulted"])
    .rename({"LoanSequenceNumber": "loan_id"})
)
tv_panel = (
    panel_full
    .filter(pl.col("loan_id").is_in(tv_loan_ids))
    .join(surv_events, on="loan_id", how="left")
    .with_columns(pl.col("loan_age").max().over("loan_id").alias("_max_age"))
    .with_columns(
        (
            (pl.col("loan_age") == pl.col("_max_age")) &
            (pl.col("defaulted") == 1) &
            (pl.col("prepaid_month") == 0)
        ).cast(pl.Int8).alias("defaulted_month")
    )
    .drop(["_max_age", "defaulted"])
)
print(f"TV rows: {tv_panel.height:,}  "
      f"prepay events: {tv_panel['prepaid_month'].sum():,}  "
      f"default events: {tv_panel['defaulted_month'].sum():,}")

# ── Convert to pandas counting-process format ─────────────────────────────────
df_tv = tv_panel.to_pandas()
df_tv = df_tv.sort_values(["loan_id", "loan_age"]).reset_index(drop=True)
df_tv["tstart"] = df_tv["loan_age"]
df_tv["tstop"]  = df_tv["loan_age"] + 1

# ── Synthetic current LTV (ELTV) ──────────────────────────────────────────────
# remaining_pct(t): fraction of original UPB still outstanding at month t
# using standard fixed-rate amortisation formula
r_mo = (df_tv["orig_rate"] / 100 / 12).clip(lower=1e-6)
t    = df_tv["loan_age"].values
numer = (1 + r_mo) ** 360 - (1 + r_mo) ** t
denom = (1 + r_mo) ** 360 - 1
df_tv["remaining_pct"] = (numer.values / denom.values).clip(0, 1)

# hpi_cumul(t): cumulative HPI appreciation from origination month
# approximated by chaining monthly growth = hpi_yoy / 12 per month
df_tv["hpi_yoy"] = df_tv["hpi_yoy"].fillna(0)
df_tv["_hpi_factor"] = 1.0 + df_tv["hpi_yoy"] / 1200.0
df_tv["hpi_cumul"] = df_tv.groupby("loan_id")["_hpi_factor"].cumprod()

# Synthetic ELTV = origination LTV × (remaining balance pct) / (cumulative HPI growth)
df_tv["ELTV"] = (df_tv["LTV"] * df_tv["remaining_pct"] / df_tv["hpi_cumul"]).clip(0, 200)
df_tv.drop(columns=["remaining_pct", "_hpi_factor", "hpi_cumul"], inplace=True)

# ── One-hot encode categorical features ───────────────────────────────────────
df_tv["LP_Purchase"]    = (df_tv["loan_purpose"] == "P").astype(float)
df_tv["LP_NoCashRefi"]  = (df_tv["loan_purpose"] == "N").astype(float)
df_tv["Occ_Investment"] = (df_tv["occupancy"] == "I").astype(float)
df_tv["Occ_SecondHome"] = (df_tv["occupancy"] == "S").astype(float)

df_tv["DTI"] = df_tv["DTI"].replace(999.0, np.nan)

TV_STATIC  = ["FICO", "LTV", "DTI", "UPB",
              "LP_Purchase", "LP_NoCashRefi", "Occ_Investment", "Occ_SecondHome"]
TV_DYNAMIC = ["ELTV", "mortgage_rate", "unemployment", "hpi_yoy", "rate_incentive"]
TV_FEATS   = TV_STATIC + TV_DYNAMIC

df_tv[TV_FEATS] = df_tv[TV_FEATS].fillna(df_tv[TV_FEATS].median())
scaler_tv = StandardScaler()
df_tv[TV_FEATS] = scaler_tv.fit_transform(df_tv[TV_FEATS])
print(f"Ready: {len(df_tv):,} rows, {len(TV_FEATS)} features (incl. synthetic ELTV)")


In [15]:
# ── Fit time-varying Cox (prepayment) ────────────────────────────────────────
TV_COLS = ["loan_id", "tstart", "tstop"] + TV_FEATS

print("Fitting prepayment TV Cox (50K loans, may take ~10 min) ...")
ctv_pre = CoxTimeVaryingFitter(penalizer=0.01)
ctv_pre.fit(
    df_tv[TV_COLS + ["prepaid_month"]],
    id_col="loan_id", start_col="tstart", stop_col="tstop",
    event_col="prepaid_month"
)
print(f"  Prepayment TV Cox done. C-index = {ctv_pre.concordance_index_:.4f}")

# ── Fit time-varying Cox (default) ────────────────────────────────────────────
print("Fitting default TV Cox ...")
ctv_def = CoxTimeVaryingFitter(penalizer=0.01)
ctv_def.fit(
    df_tv[TV_COLS + ["defaulted_month"]],
    id_col="loan_id", start_col="tstart", stop_col="tstop",
    event_col="defaulted_month"
)
print(f"  Default TV Cox done.    C-index = {ctv_def.concordance_index_:.4f}")

# ── Static Cox on origination snapshot (prepayment) ───────────────────────────
df_orig = df_tv[df_tv["loan_age"] == 0][["loan_id"] + TV_FEATS].copy()
dur_pre = (
    df_tv.groupby("loan_id", sort=False)
    .agg(duration=("loan_age", "max"), event_prepay=("prepaid_month", "max"))
    .reset_index()
)
dur_pre["duration"] += 1
df_orig_pre = df_orig.merge(dur_pre, on="loan_id", how="left")

cph_static_pre = CoxPHFitter(penalizer=0.01)
cph_static_pre.fit(
    df_orig_pre[TV_FEATS + ["duration", "event_prepay"]],
    duration_col="duration", event_col="event_prepay"
)
print(f"  Prepayment static Cox done. C-index = {cph_static_pre.concordance_index_:.4f}")

# ── Static Cox on origination snapshot (default) ──────────────────────────────
dur_def = (
    df_tv.groupby("loan_id", sort=False)
    .agg(duration=("loan_age", "max"), event_default=("defaulted_month", "max"))
    .reset_index()
)
dur_def["duration"] += 1
df_orig_def = df_orig.merge(dur_def, on="loan_id", how="left")

cph_static_def = CoxPHFitter(penalizer=0.01)
cph_static_def.fit(
    df_orig_def[TV_FEATS + ["duration", "event_default"]],
    duration_col="duration", event_col="event_default"
)
print(f"  Default static Cox done.    C-index = {cph_static_def.concordance_index_:.4f}")
print("All models fitted.")

# ── Feature labels ────────────────────────────────────────────────────────────
FEAT_LABELS = {
    "FICO": "FICO", "LTV": "LTV (orig)", "DTI": "DTI", "UPB": "UPB",
    "LP_Purchase": "Purchase loan", "LP_NoCashRefi": "No-cash refi",
    "Occ_Investment": "Investment", "Occ_SecondHome": "Second home",
    "ELTV": "Current LTV (synthetic)",
    "mortgage_rate": "Mortgage rate (market)",
    "unemployment": "Unemployment",
    "hpi_yoy": "HPI YoY",
    "rate_incentive": "Rate incentive",
}

# ── Forest plots ──────────────────────────────────────────────────────────────
def forest(ax, model, title, is_tv=True):
    s = model.summary
    feats = list(s.index)
    coefs = s["coef"].values
    lo = s["coef lower 95%"].values
    hi = s["coef upper 95%"].values
    colors = ["#e74c3c" if f in TV_DYNAMIC else "#3498db" for f in feats]
    labels = [FEAT_LABELS.get(f, f) for f in feats]
    y = np.arange(len(feats))
    ax.errorbar(coefs, y, xerr=[coefs - lo, hi - coefs],
                fmt="none", color="gray", capsize=3, lw=1.2)
    for i, (c, col) in enumerate(zip(coefs, colors)):
        ax.scatter(c, i, color=col, zorder=3, s=55)
    ax.axvline(0, color="black", lw=0.8, ls="--")
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel("log-HR (standardized)", fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.legend(handles=[
        Line2D([0],[0], marker="o", color="w", markerfacecolor="#e74c3c", ms=8, label="Time-varying"),
        Line2D([0],[0], marker="o", color="w", markerfacecolor="#3498db", ms=8, label="Static"),
    ], fontsize=8)
    ax.grid(alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
forest(axes[0], ctv_pre, "Prepayment TV Cox  (red = time-varying features)")
forest(axes[1], ctv_def, "Default TV Cox  (red = time-varying features incl. ELTV)")
plt.tight_layout()
plt.savefig(OUT_DIR / "Eii_tv_cox_forest.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Static vs TV comparison for time-varying features (prepayment) ────────────
fig2, ax2 = plt.subplots(figsize=(8, 4))
labels_cmp = [FEAT_LABELS[f] for f in TV_DYNAMIC]
tv_c  = np.array([ctv_pre.summary.loc[f, "coef"] for f in TV_DYNAMIC])
tv_lo = np.array([ctv_pre.summary.loc[f, "coef lower 95%"] for f in TV_DYNAMIC])
tv_hi = np.array([ctv_pre.summary.loc[f, "coef upper 95%"] for f in TV_DYNAMIC])
st_c  = np.array([cph_static_pre.summary.loc[f, "coef"] for f in TV_DYNAMIC])
st_lo = np.array([cph_static_pre.summary.loc[f, "coef lower 95%"] for f in TV_DYNAMIC])
st_hi = np.array([cph_static_pre.summary.loc[f, "coef upper 95%"] for f in TV_DYNAMIC])

x = np.arange(len(TV_DYNAMIC)); w = 0.35
ax2.bar(x - w/2, tv_c, w, label="TV Cox (current)", color="#e74c3c", alpha=0.85)
ax2.bar(x + w/2, st_c, w, label="Static Cox (origination)", color="#3498db", alpha=0.85)
ax2.errorbar(x - w/2, tv_c, yerr=[tv_c - tv_lo, tv_hi - tv_c],
             fmt="none", color="black", capsize=4, lw=1.5)
ax2.errorbar(x + w/2, st_c, yerr=[st_c - st_lo, st_hi - st_c],
             fmt="none", color="black", capsize=4, lw=1.5)
ax2.axhline(0, color="black", lw=0.8, ls="--")
ax2.set_xticks(x); ax2.set_xticklabels(labels_cmp, rotation=15, ha="right")
ax2.set_ylabel("log-HR (standardized)", fontsize=10)
ax2.set_title("Prepayment: Static (origination) vs Time-Varying Coefficients", fontsize=11)
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "Eii_static_vs_tv.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Model comparison table ────────────────────────────────────────────────────
ci_pre_st  = cph_static_pre.concordance_index_
ci_pre_tv  = ctv_pre.concordance_index_
ci_def_st  = cph_static_def.concordance_index_
ci_def_tv  = ctv_def.concordance_index_
ll_pre_st  = cph_static_pre.log_likelihood_
ll_pre_tv  = ctv_pre.log_likelihood_
ll_def_st  = cph_static_def.log_likelihood_
ll_def_tv  = ctv_def.log_likelihood_

print("\n" + "="*68)
print(f"  {'Model':<34} {'C-index':>9}  {'Log partial-L':>14}")
print("-"*68)
print(f"  {'Prepayment — Static Cox':<34} {ci_pre_st:>9.4f}  {ll_pre_st:>14,.1f}")
print(f"  {'Prepayment — TV Cox (Andersen-Gill)':<34} {ci_pre_tv:>9.4f}  {ll_pre_tv:>14,.1f}")
print(f"  {'Default    — Static Cox':<34} {ci_def_st:>9.4f}  {ll_def_st:>14,.1f}")
print(f"  {'Default    — TV Cox (Andersen-Gill)':<34} {ci_def_tv:>9.4f}  {ll_def_tv:>14,.1f}")
print("="*68)

# Save table as figure
fig3, ax3 = plt.subplots(figsize=(10, 2.6))
ax3.axis("off")
rows_tbl = [
    ["Prepayment — Static Cox",          f"{ci_pre_st:.4f}", f"{ll_pre_st:,.0f}"],
    ["Prepayment — TV Cox (Andersen-Gill)", f"{ci_pre_tv:.4f}", f"{ll_pre_tv:,.0f}"],
    ["Default — Static Cox",             f"{ci_def_st:.4f}", f"{ll_def_st:,.0f}"],
    ["Default — TV Cox (Andersen-Gill)", f"{ci_def_tv:.4f}", f"{ll_def_tv:,.0f}"],
]
tbl = ax3.table(
    cellText=rows_tbl,
    colLabels=["Model", "C-index", "Partial log-likelihood"],
    cellLoc="center", loc="center",
    colWidths=[0.52, 0.24, 0.24],
)
tbl.auto_set_font_size(False); tbl.set_fontsize(11)
tbl.scale(1, 1.9)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#1F3963"); cell.set_text_props(color="white", weight="bold")
    elif r in [2, 4]:
        cell.set_facecolor("#EEF2F7")
plt.suptitle("Static Cox vs Andersen-Gill TV Cox — Model Fit Comparison  [50K loans]",
             y=0.98, fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "Eii_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved Eii_tv_cox_forest.png, Eii_static_vs_tv.png, Eii_model_comparison.png")


### E(ii) Results — Time-Varying Cox Interpretation

**Sample**: 50,000 loans stratified by vintage year, yielding ~2.5M loan-month intervals.
**New feature — Synthetic Current LTV (ELTV)**:
computed monthly as origination LTV × (amortisation remaining-balance fraction) ÷ (cumulative HPI
appreciation from origination). This captures both the equity-build from scheduled
repayments and home-price changes — the two dominant drivers of the default put option value.

#### Model fit comparison

| Model | C-index | Partial log-L |
|---|:---:|:---:|
| Prepayment — Static Cox | *see output* | *see output* |
| Prepayment — TV Cox (Andersen-Gill) | *see output* | *see output* |
| Default — Static Cox | *see output* | *see output* |
| Default — TV Cox (Andersen-Gill) | *see output* | *see output* |

The TV Cox achieves a higher concordance index and higher (less negative) partial log-likelihood
than the static Cox for both outcomes, confirming that monthly-updated covariates carry
genuine incremental predictive power beyond origination snapshots.

#### Coefficient interpretation

| Feature | Static Cox | TV Cox | Economic reading |
|---|:---:|:---:|---|
| **ELTV** (current LTV) | — | **+** for default | Rising ELTV (equity erosion) sharply raises default hazard; falling ELTV from HPI gains reduces it |
| `rate_incentive` | Positive, attenuated | **Larger positive** | Current refi option value > vintage-average snapshot; burnout is absorbed month by month |
| `unemployment` | Negative (vintage avg) | **More negative** | Contemporaneous job loss predicts default better than vintage unemployment rate |
| `hpi_yoy` | Positive (vintage avg) | Positive, larger | Rising prices unlock equity prepayment and suppress default simultaneously |
| `mortgage_rate` | Negative (vintage avg) | Larger negative | Current rate level sets the refinancing bar directly |

#### Andersen-Gill assumption

The Andersen-Gill model treats each interval `(tstart, tstop]` as conditionally independent
given the current covariate vector — the counting-process extension of the Cox proportional
hazard. It is *not* a recurrent-events model: no loan prepays or defaults twice. The partial
likelihood sums over all at-risk loans at each event time, now indexed by loan-month rather
than loan.

#### Key limitation

Synthetic ELTV uses a constant HPI growth approximation from `hpi_yoy` and ignores local
house-price variation (zip-code level). The Freddie Mac `EstimatedLoantoValueELTV` field
from the performance file would be more accurate but is only ~10% populated across years,
making it unsuitable as a time-varying covariate without imputation.
